# WorldSim DGX Spark QLoRA Baseline Training

Thin Jupyter wrapper around the shared WorldSim baseline trainer. This notebook runs baseline fine-tuning, inspects metrics and generations, records run metadata, and selects a candidate adapter without duplicating trainer logic.


## 1. Environment Check

Verify CUDA visibility, Python environment, and key training libraries before launching baseline training.


In [ ]:
from pathlib import Path
import importlib
import sys

import torch

cwd = Path.cwd()
repo_marker = cwd / 'training/lib/qlora_smoke.py'
assert repo_marker.exists(), f'Run this notebook from the repo root. cwd={cwd}'

library_summary = {}
for name in ('transformers', 'peft', 'trl', 'datasets'):
    try:
        module = importlib.import_module(name)
        library_summary[name] = {'available': True, 'version': getattr(module, '__version__', 'unknown')}
    except Exception as exc:  # noqa: BLE001
        library_summary[name] = {'available': False, 'error': f'{type(exc).__name__}: {exc}'}

{
    'python_executable': sys.executable,
    'cwd': str(cwd),
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'gpu_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
    'torch_version': torch.__version__,
    'torch_cuda_version': getattr(torch.version, 'cuda', None),
    'libraries': library_summary,
}


## 2. Config

Edit this single cell to switch between dry-run and real baseline runs.


In [ ]:
from pathlib import Path

from training.lib.qlora_smoke import BASELINE_MODEL_NAME

CONFIG = {
    'run_id': 'run-001',
    'dry_run': True,
    'model_name': BASELINE_MODEL_NAME,
    'dataset': 'worldsim-v31-mix-v1',
    'train_file': Path('data/training/worldsim-v31-mix-v1/train_converted.jsonl'),
    'dev_file': Path('data/training/worldsim-v31-mix-v1/dev_converted.jsonl'),
    'max_steps': 200,
    'max_train_samples': 0,
    'max_eval_samples': 0,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1e-4,
    'logging_steps': 5,
    'eval_steps': 25,
    'save_steps': 25,
    'save_total_limit': 2,
    'require_qlora': True,
    'seed': 42,
}
CONFIG['output_dir'] = Path('outputs/baseline/worldsim-v31-mix-v1') / CONFIG['run_id']
CONFIG


## 3. Dataset Preview

Inspect a few training rows to confirm the chat-template contract before starting.


In [ ]:
from scripts.common import read_jsonl

train_preview_rows = read_jsonl(CONFIG['train_file'])[:3]
dataset_preview = [
    {
        'task': row.get('task'),
        'system': row['messages'][0]['content'],
        'user': row['messages'][1]['content'],
        'assistant': row['messages'][-1]['content'],
    }
    for row in train_preview_rows
]
dataset_preview


## 4. Trainer Invocation

Call the shared baseline trainer directly. No trainer logic is duplicated here.


In [ ]:
import time

from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

cfg = SmokeRunConfig(
    run_mode='baseline',
    model_name=CONFIG['model_name'],
    train_file=CONFIG['train_file'],
    dev_file=CONFIG['dev_file'],
    output_dir=CONFIG['output_dir'],
    max_steps=CONFIG['max_steps'],
    max_train_samples=CONFIG['max_train_samples'],
    max_eval_samples=CONFIG['max_eval_samples'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    logging_steps=CONFIG['logging_steps'],
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    require_qlora=CONFIG['require_qlora'],
    seed=CONFIG['seed'],
    dry_run=CONFIG['dry_run'],
)

started_at = time.perf_counter()
result = run_baseline_or_raise(cfg)
elapsed_seconds = round(time.perf_counter() - started_at, 2)
{'elapsed_seconds': elapsed_seconds, 'result': result.to_dict()}


## 5. Training Metrics

Plot step-wise losses if a trainer state exists; otherwise fall back to final train/eval losses from the shared metrics artifact.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

metrics_payload = None
trainer_state = None
metrics_path = Path(result.metrics_path) if result.metrics_path else None
trainer_state_path = Path(result.output_dir) / 'checkpoints' / 'trainer_state.json'
if metrics_path and metrics_path.exists():
    metrics_payload = json.loads(metrics_path.read_text(encoding='utf-8'))
if trainer_state_path.exists():
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8'))

if trainer_state and trainer_state.get('log_history'):
    train_points = [(entry['step'], entry['loss']) for entry in trainer_state['log_history'] if 'loss' in entry]
    eval_points = [(entry['step'], entry['eval_loss']) for entry in trainer_state['log_history'] if 'eval_loss' in entry]
    plt.figure(figsize=(8, 4))
    if train_points:
        plt.plot([step for step, _ in train_points], [loss for _, loss in train_points], label='train_loss')
    if eval_points:
        plt.plot([step for step, _ in eval_points], [loss for _, loss in eval_points], label='eval_loss')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.title('WorldSim baseline training losses')
    plt.legend()
    plt.show()
else:
    final_metrics = {
        'train_loss': result.train_loss,
        'eval_loss': result.eval_loss,
    }
    plt.figure(figsize=(5, 4))
    plt.bar(final_metrics.keys(), [value if value is not None else 0.0 for value in final_metrics.values()])
    plt.title('Final losses (no trainer_state log history found)')
    plt.show()
    final_metrics


## 6. Sample Generation

Load the trainer-produced sample generations and inspect a few outputs, with special attention to Task G.


In [ ]:
from training.lib.qlora_smoke import load_sample_generations

sample_rows = load_sample_generations(result.output_dir) if result.sample_path else []
sample_preview = sample_rows[:5]
task_g_preview = [row for row in sample_rows if row.get('task') == 'G'][:3]
{
    'sample_count': len(sample_rows),
    'sample_preview': sample_preview,
    'task_g_preview': task_g_preview,
}


## 7. Analyzer Integration

Run the shared analyzer against `sample_generations.jsonl` and surface structure / semantics / retry-related signals.


In [ ]:
import json

from tools.generation_analyzer import generate_report, recommend_next_action

analysis_report = generate_report(sample_rows, examples_per_category=3) if sample_rows else None
analysis_report_path = Path(result.output_dir) / 'analysis_report.json'
if analysis_report is not None:
    analysis_report_path.write_text(json.dumps(analysis_report, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_recommendation = recommend_next_action(analysis_report) if analysis_report else None
retry_samples = sum(1 for row in sample_rows if int(row.get('structured_attempt_count', 1) or 1) > 1)
retry_rate = round(retry_samples / len(sample_rows), 4) if sample_rows else None
{
    'analysis_report': analysis_report,
    'analysis_report_path': str(analysis_report_path) if analysis_report is not None else None,
    'retry_samples': retry_samples,
    'retry_rate': retry_rate,
    'recommended_next_action': analysis_recommendation,
}


## 8. Model Registry (Simple Local)

Append successful baseline runs to a lightweight local registry for later comparison.


In [ ]:
import json

registry_path = Path('outputs/model_registry.json')
registry = {'runs': []}
if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))

registry_entry = {
    'run_id': CONFIG['run_id'],
    'mode': 'baseline',
    'status': result.status,
    'model': CONFIG['model_name'],
    'dataset': CONFIG['dataset'],
    'steps': CONFIG['max_steps'],
    'output_dir': result.output_dir,
    'adapter_path': result.adapter_dir,
    'sample_path': result.sample_path,
    'analysis_report_path': str(analysis_report_path) if analysis_report is not None else None,
    'metrics': {
        'train_loss': result.train_loss,
        'eval_loss': result.eval_loss,
        'semantic_low_quality': analysis_report.get('semantic_low_quality_count') if analysis_report else None,
        'malformed_json': analysis_report.get('malformed_json_count') if analysis_report else None,
        'retry_rate': retry_rate,
    },
}

existing_runs = [run for run in registry.get('runs', []) if run.get('run_id') != CONFIG['run_id']]
if result.adapter_dir:
    existing_runs.append(registry_entry)
    registry['runs'] = existing_runs
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    registry_path.write_text(json.dumps(registry, ensure_ascii=False, indent=2), encoding='utf-8')

{
    'registry_path': str(registry_path),
    'registry_entry': registry_entry,
    'registered_runs': len(registry.get('runs', [])),
}


## 9. Candidate Selection

Pick the best local adapter using a conservative rule: lower `semantic_low_quality`, then lower `eval_loss`.


In [ ]:
import math

best_adapter_path = Path('outputs/best_adapter.txt')
candidate_runs = [run for run in registry.get('runs', []) if run.get('status') == 'ok' and run.get('adapter_path')]

def candidate_score(run):
    metrics = run.get('metrics', {})
    semantic_low_quality = metrics.get('semantic_low_quality')
    eval_loss = metrics.get('eval_loss')
    semantic_score = semantic_low_quality if semantic_low_quality is not None else math.inf
    eval_score = eval_loss if eval_loss is not None else math.inf
    return (semantic_score, eval_score, run.get('run_id'))

best_run = min(candidate_runs, key=candidate_score) if candidate_runs else None
if best_run is not None:
    best_adapter_path.parent.mkdir(parents=True, exist_ok=True)
    best_adapter_path.write_text(str(best_run['adapter_path']), encoding='utf-8')

{
    'best_run': best_run,
    'best_adapter_pointer': str(best_adapter_path),
}


## 10. Inference Test

Use the selected adapter's own generated samples as the first inference sanity check for Tasks A, B, and G.


In [ ]:
import json

from scripts.common import read_jsonl
from tools.generation_analyzer import generate_report

best_run_samples = []
best_run_report = None
if best_run is not None and best_run.get('sample_path'):
    best_run_samples = read_jsonl(Path(best_run['sample_path']))
    abg_samples = [row for row in best_run_samples if row.get('task') in {'A', 'B', 'G'}]
    best_run_report = generate_report(abg_samples, examples_per_category=2)
    retry_count = sum(1 for row in abg_samples if int(row.get('structured_attempt_count', 1) or 1) > 1)
    retry_rate = round(retry_count / len(abg_samples), 4) if abg_samples else None
    output = {
        'best_adapter': best_run['adapter_path'],
        'task_a_b_g_examples': abg_samples[:6],
        'best_run_report': best_run_report,
        'retry_rate': retry_rate,
    }
else:
    output = {
        'best_adapter': None,
        'task_a_b_g_examples': [],
        'best_run_report': None,
        'retry_rate': None,
    }
output
